In [1]:
import pandas as pd
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================
# Define the base project directory
BASE_PATH = '..' 

# Input and output file paths
INPUT_CSV = os.path.join(BASE_PATH, 'includes', 'dados','Tabela_consumo_Itapua_120m.csv')
OUTPUT_CSV = os.path.join(BASE_PATH, 'includes', 'Tabela_consumo_real_historico_2016_2025.csv')

def main():
    print("--- Generating Historical Ground Truth (2016-2025) ---")

    # ==========================================
    # 2. Data Loading and Filtering
    # ==========================================
    if not os.path.exists(INPUT_CSV):
        print(f"Error: Input file not found at {INPUT_CSV}")
        return

    # Load the CSV file
    df = pd.read_csv(INPUT_CSV, delimiter=';')

    # Ensure reference month is integer for filtering
    df['AM_REFERENCIA'] = df['AM_REFERENCIA'].astype(int)

    # Filter for the validation period: January 2016 to December 2025
    # This matches the period you will run in the GAMA hindcasting simulation
    mask = (df['AM_REFERENCIA'] >= 201601) & (df['AM_REFERENCIA'] <= 202512)
    df_validation = df.loc[mask].copy()

    print(f"Total records found for the validation period: {len(df_validation)}")

    if df_validation.empty:
        print("Warning: No records found for the specified period.")
        return

    # ==========================================
    # 3. Aggregating Total Monthly Consumption
    # ==========================================
    # We sum the consumption (HCLQTCON) of all households for each month
    # This represents the total demand observed in Itapuã
    historical_trend = df_validation.groupby('AM_REFERENCIA')['HCLQTCON'].sum().reset_index()

    # Rename columns for clarity in the final comparison plot
    historical_trend.rename(columns={'HCLQTCON': 'real_total_consumption'}, inplace=True)

    # Sort by date to ensure a proper time-series line
    historical_trend = historical_trend.sort_values('AM_REFERENCIA')

    # ==========================================
    # 4. Saving Results
    # ==========================================
    historical_trend.to_csv(OUTPUT_CSV, index=False, sep=';')

    print(f"Success! Historical Ground Truth saved to: {OUTPUT_CSV}")
    print(f"Total months processed: {len(historical_trend)}")

if __name__ == "__main__":
    main()

--- Generating Historical Ground Truth (2016-2025) ---


Total records found for the validation period: 1908153
Success! Historical Ground Truth saved to: ..\includes\Tabela_consumo_real_historico_2016_2025.csv
Total months processed: 120
